# DRAgent: Interactive Exploration

This notebook provides an interactive way to explore the DRAgent's capabilities.

In [ ]:
# Setup
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from dr_agent import create_dr_agent, create_baseline_llm, run_baseline_recommendation
from dr_agent import fetch_sdge_prices, fetch_caiso_carbon, solve_dr_optimization

# Set your API key
os.environ['ANTHROPIC_API_KEY'] = 'your-key-here'  # Replace with your key

print("✓ Setup complete!")

## 1. Visualize Price and Carbon Data

In [ ]:
# Fetch data
prices_json = fetch_sdge_prices.invoke({})
carbon_json = fetch_caiso_carbon.invoke({})

prices = json.loads(prices_json)
carbon = json.loads(carbon_json)

# Extract arrays
hours = list(range(24))
price_values = [p['price_per_kwh'] for p in prices['prices']]
carbon_values = [c['carbon_intensity_lbs_per_mwh'] for c in carbon['carbon_data']]

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Price plot
ax1.plot(hours, price_values, marker='o', linewidth=2, markersize=6)
ax1.fill_between(hours, price_values, alpha=0.3)
ax1.set_xlabel('Hour of Day', fontsize=12)
ax1.set_ylabel('Price ($/kWh)', fontsize=12)
ax1.set_title('SDG&E Time-of-Use Electricity Prices', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.set_xticks(range(0, 24, 2))

# Highlight periods
ax1.axvspan(0, 6, alpha=0.2, color='green', label='Super Off-Peak')
ax1.axvspan(16, 21, alpha=0.2, color='red', label='On-Peak')
ax1.legend()

# Carbon plot
ax2.plot(hours, carbon_values, marker='s', linewidth=2, markersize=6, color='brown')
ax2.fill_between(hours, carbon_values, alpha=0.3, color='brown')
ax2.set_xlabel('Hour of Day', fontsize=12)
ax2.set_ylabel('Carbon Intensity (lbs/MWh)', fontsize=12)
ax2.set_title('CAISO Grid Carbon Intensity', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_xticks(range(0, 24, 2))

# Highlight clean periods
ax2.axvspan(10, 16, alpha=0.2, color='green', label='Solar Peak (Clean)')
ax2.axvspan(16, 22, alpha=0.2, color='red', label='Evening Peak (Dirty)')
ax2.legend()

plt.tight_layout()
plt.show()

print(f"Cheapest hour: {hours[np.argmin(price_values)]}:00 (${min(price_values):.2f}/kWh)")
print(f"Most expensive hour: {hours[np.argmax(price_values)]}:00 (${max(price_values):.2f}/kWh)")
print(f"Cleanest hour: {hours[np.argmin(carbon_values)]}:00 ({min(carbon_values)} lbs/MWh)")
print(f"Dirtiest hour: {hours[np.argmax(carbon_values)]}:00 ({max(carbon_values)} lbs/MWh)")

## 2. Test the Agent

In [ ]:
# Create agent
agent = create_dr_agent()

# Example query
query = """
Help me schedule my appliances for tomorrow to save money:

- EV: needs 16 kWh, available 10 PM to 7 AM, max 11 kW
- Dishwasher: needs 3.6 kWh, available 8 PM to 11 PM, max 2 kW

Household peak: 15 kW
"""

# Run agent
result = agent.invoke({"input": query})
print(result["output"])

## 3. Compare Optimal vs Baseline Schedule

In [ ]:
# Define appliances
appliances = [
    {
        "name": "EV",
        "energy_required_kwh": 16.0,
        "start_hour": 22,
        "end_hour": 7,
        "min_power_kw": 0.0,
        "max_power_kw": 11.0,
        "household_peak_limit": 15.0
    },
    {
        "name": "Dishwasher",
        "energy_required_kwh": 3.6,
        "start_hour": 20,
        "end_hour": 23,
        "min_power_kw": 0.0,
        "max_power_kw": 2.0,
        "household_peak_limit": 15.0
    }
]

# Solve optimization
opt_result = solve_dr_optimization.invoke({
    "appliances_json": json.dumps(appliances),
    "prices_json": prices_json,
    "carbon_json": carbon_json,
    "optimization_goal": "cost"
})

result = json.loads(opt_result)

if result['status'] == 'success':
    # Extract schedules
    ev_schedule = result['schedule']['EV']['hourly_consumption_kwh']
    dish_schedule = result['schedule']['Dishwasher']['hourly_consumption_kwh']
    
    # Create baseline (run at earliest time)
    ev_baseline = [0] * 24
    ev_baseline[22] = 11  # Start at 10 PM
    ev_baseline[23] = 5   # Finish by 11 PM
    
    dish_baseline = [0] * 24
    dish_baseline[20] = 2  # Start at 8 PM
    dish_baseline[21] = 1.6
    
    # Plot comparison
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    
    # EV - Baseline
    ax1.bar(hours, ev_baseline, alpha=0.7, color='red', label='EV')
    ax1.set_title('Baseline Schedule (No Optimization)', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Power (kW)', fontsize=10)
    ax1.set_xlabel('Hour', fontsize=10)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # EV - Optimized
    ax2.bar(hours, ev_schedule, alpha=0.7, color='green', label='EV')
    ax2.set_title('Optimized Schedule (DRAgent)', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Power (kW)', fontsize=10)
    ax2.set_xlabel('Hour', fontsize=10)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Cost comparison
    baseline_cost = sum(p * (b + d) for p, b, d in zip(price_values, ev_baseline, dish_baseline))
    optimized_cost = sum(p * (o + d) for p, o, d in zip(price_values, ev_schedule, dish_schedule))
    
    ax3.bar(['Baseline', 'Optimized'], [baseline_cost, optimized_cost], color=['red', 'green'])
    ax3.set_title('Total Cost Comparison', fontsize=12, fontweight='bold')
    ax3.set_ylabel('Cost ($)', fontsize=10)
    ax3.grid(True, alpha=0.3)
    
    # Add savings annotation
    savings = baseline_cost - optimized_cost
    ax3.text(0.5, max(baseline_cost, optimized_cost) * 0.5, 
             f'Savings: ${savings:.2f}\n({100*savings/baseline_cost:.1f}%)',
             ha='center', fontsize=12, bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))
    
    # Hourly cost breakdown
    baseline_hourly_cost = [p * (b + d) for p, b, d in zip(price_values, ev_baseline, dish_baseline)]
    optimized_hourly_cost = [p * (o + d) for p, o, d in zip(price_values, ev_schedule, dish_schedule)]
    
    ax4.plot(hours, baseline_hourly_cost, marker='o', label='Baseline', color='red')
    ax4.plot(hours, optimized_hourly_cost, marker='s', label='Optimized', color='green')
    ax4.set_title('Hourly Cost Breakdown', fontsize=12, fontweight='bold')
    ax4.set_ylabel('Cost ($)', fontsize=10)
    ax4.set_xlabel('Hour', fontsize=10)
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print metrics
    print("\n" + "="*60)
    print("OPTIMIZATION RESULTS")
    print("="*60)
    print(f"Baseline Cost: ${baseline_cost:.2f}")
    print(f"Optimized Cost: ${optimized_cost:.2f}")
    print(f"Savings: ${savings:.2f} ({100*savings/baseline_cost:.1f}%)")
    print(f"\nMonthly Savings: ${savings * 30:.2f}")
    print(f"Annual Savings: ${savings * 365:.2f}")
    print("="*60)

## 4. Test Baseline LLM (No Tools)

In [ ]:
# Create baseline LLM
baseline_llm = create_baseline_llm()

# Same query
baseline_result = run_baseline_recommendation(baseline_llm, query)

print("BASELINE LLM RESPONSE (No Tools):")
print("="*60)
print(baseline_result)
print("="*60)
print("\nNote: Baseline relies on general knowledge, not real data or optimization")

## 5. Custom Scenario Testing

In [ ]:
# Try your own scenario!
custom_query = """
I want to minimize my carbon footprint tomorrow.

Appliances:
- EV: needs 20 kWh, available 9 PM to 8 AM, max 11 kW
- Dryer: needs 4.5 kWh, available 9 PM to 1 AM, max 4 kW
- Pool Pump: needs 8 kWh, can run anytime, max 2 kW

Household peak: 15 kW
"""

result = agent.invoke({"input": custom_query})
print(result["output"])

## 6. Failure Case Testing

In [ ]:
# Test with missing information
incomplete_query = "Help me charge my EV to save money"

print("Testing incomplete query...\n")
result = agent.invoke({"input": incomplete_query})
print(result["output"])
print("\n" + "="*60)
print("Expected: Agent should request missing information")